In [0]:
%pip install openpyxl


Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
raw_member_path = "abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/"

display(dbutils.fs.ls(raw_member_path))

path,name,size,modificationTime
abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/AUS.xlsx,AUS.xlsx,15476,1795191819000
abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/IND.xlsx,IND.xlsx,10285,1795191819000
abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/USA.xlsx,USA.xlsx,10192,1795191819000


In [0]:
from io import BytesIO
import pandas as pd

def read_excel_file(file_path):
    binary_data = (
        spark.read.format("binaryFile")
        .load(file_path)
        .first()["content"]
    )

    pandas_df = pd.read_excel(
        BytesIO(binary_data),
        dtype=str,
        keep_default_na=False
    )

    return spark.createDataFrame(pandas_df)

In [0]:
raw_member_path = "abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/"

In [0]:
display(dbutils.fs.ls(raw_member_path))

path,name,size,modificationTime
abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/AUS.xlsx,AUS.xlsx,15476,1795191819000
abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/IND.xlsx,IND.xlsx,10285,1795191819000
abfss://raw@incubytedatalake01.dfs.core.windows.net/Member/USA.xlsx,USA.xlsx,10192,1795191819000


In [0]:
aus_path = raw_member_path + "AUS.xlsx"

aus_df = read_excel_file(aus_path)

display(aus_df)

Unique ID,Member Name,Tier Type,Date of Birth,Date of Enrollment,Date of Flight
1,Mike,PLT,NULL,2022-05-11 00:00:00,2022-08-01 00:00:00
2,Jonnathan,GLD,1997-12-13 00:00:00,2021-13-13,2022-01-05 00:00:00
3,Cristina,SLV,1998-03-12 00:00:00,2022-03-12 00:00:00,2022-03-20 00:00:00


In [0]:
from io import BytesIO
import pandas as pd

def read_excel_file(file_path):
    binary_data = (
        spark.read.format("binaryFile")
        .load(file_path)
        .first()["content"]
    )

    pandas_df = pd.read_excel(
        BytesIO(binary_data),
        dtype=str,
        keep_default_na=False
    )

    return spark.createDataFrame(pandas_df)

aus_df = read_excel_file(raw_member_path + "AUS.xlsx")

display(aus_df)

Unique ID,Member Name,Tier Type,Date of Birth,Date of Enrollment,Date of Flight
1,Mike,PLT,NULL,2022-05-11 00:00:00,2022-08-01 00:00:00
2,Jonnathan,GLD,1997-12-13 00:00:00,2021-13-13,2022-01-05 00:00:00
3,Cristina,SLV,1998-03-12 00:00:00,2022-03-12 00:00:00,2022-03-20 00:00:00


In [0]:
# Read India Excel file
ind_df = read_excel_file(raw_member_path + "IND.xlsx")

# Read USA Excel file
usa_df = read_excel_file(raw_member_path + "USA.xlsx")

# Display the datasets
display(ind_df)
display(usa_df)

ID,Name,DOB,TierCode,EnrollmentDate,Individual or Corporate,Flight Date
1,Vikas,1998-12-01 00:00:00,SLV,2022-01-01 00:00:00,I,2022-06-15 00:00:00
2,Rahul,1982-08-13 00:00:00,GLD,2022-03-05 00:00:00,C,2022-03-10 00:00:00
3,Sameer,1952-08-13 00:00:00,GLD,2022-02-20 00:00:00,I,2022-02-25 00:00:00


ID,Name,TierCode,EnrollmentDate,FlightDate
1,Sam,PLT,6152022,8202022
2,John,SLV,1052022,1152022
3,Mike,GLD,12282021,12302021


In [0]:
print("Australia records:", aus_df.count())
print("India records:", ind_df.count())
print("USA records:", usa_df.count())

Australia records: 3
India records: 3
USA records: 3


In [0]:
print("Australia columns:", aus_df.columns)
print("India columns:", ind_df.columns)
print("USA columns:", usa_df.columns)

Australia columns: ['Unique ID', 'Member Name', 'Tier Type', 'Date of Birth', 'Date of Enrollment', 'Date of Flight']
India columns: ['ID', 'Name', 'DOB', 'TierCode', 'EnrollmentDate', 'Individual or Corporate', 'Flight Date']
USA columns: ['ID', 'Name', 'TierCode', 'EnrollmentDate', 'FlightDate']


In [0]:
from pyspark.sql.functions import lit

# Australia
aus_standard = aus_df.selectExpr(
    "`Unique ID` as Member_ID",
    "`Member Name` as Member_Name",
    "`Tier Type` as Tier_Code",
    "`Date of Birth` as DOB",
    "`Date of Enrollment` as Enrollment_Date",
    "`Date of Flight` as Last_Flight_Date"
).withColumn("Country", lit("Australia")) \
 .withColumn("Member_Type", lit(None).cast("string"))

# India
ind_standard = ind_df.selectExpr(
    "`ID` as Member_ID",
    "`Name` as Member_Name",
    "`TierCode` as Tier_Code",
    "`DOB` as DOB",
    "`EnrollmentDate` as Enrollment_Date",
    "`Flight Date` as Last_Flight_Date",
    "`Individual or Corporate` as Member_Type"
).withColumn("Country", lit("India"))

# USA
usa_standard = usa_df.selectExpr(
    "`ID` as Member_ID",
    "`Name` as Member_Name",
    "`TierCode` as Tier_Code",
    "cast(null as string) as DOB",
    "`EnrollmentDate` as Enrollment_Date",
    "`FlightDate` as Last_Flight_Date"
).withColumn("Country", lit("USA")) \
 .withColumn("Member_Type", lit(None).cast("string"))

In [0]:
print("Australia:")
aus_standard.printSchema()

print("India:")
ind_standard.printSchema()

print("USA:")
usa_standard.printSchema()

Australia:
root
 |-- Member_ID: string (nullable = true)
 |-- Member_Name: string (nullable = true)
 |-- Tier_Code: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Enrollment_Date: string (nullable = true)
 |-- Last_Flight_Date: string (nullable = true)
 |-- Country: string (nullable = false)
 |-- Member_Type: string (nullable = true)

India:
root
 |-- Member_ID: string (nullable = true)
 |-- Member_Name: string (nullable = true)
 |-- Tier_Code: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Enrollment_Date: string (nullable = true)
 |-- Last_Flight_Date: string (nullable = true)
 |-- Member_Type: string (nullable = true)
 |-- Country: string (nullable = false)

USA:
root
 |-- Member_ID: string (nullable = true)
 |-- Member_Name: string (nullable = true)
 |-- Tier_Code: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Enrollment_Date: string (nullable = true)
 |-- Last_Flight_Date: string (nullable = true)
 |-- Country: string (nullabl

In [0]:
from pyspark.sql import functions as F

In [0]:
# --------------------------------------------------
# 1. STANDARDIZE AUSTRALIA DATA
# --------------------------------------------------

aus_standard = aus_df.select(
    F.col("Member Name").alias("Member_Name"),
    F.col("Unique ID").alias("Member_ID"),
    F.col("Date of Enrollment").alias("Enrollment_Date"),
    F.col("Date of Flight").alias("Last_Flight_Date"),
    F.col("Tier Type").alias("Tier_Code"),
    F.col("Date of Birth").alias("DOB")
)

aus_standard = (
    aus_standard
    .withColumn("Agent_Name", F.lit(None).cast("string"))
    .withColumn("State", F.lit(None).cast("string"))
    .withColumn("Country", F.lit(None).cast("string"))
    .withColumn("Post_Code", F.lit(None).cast("string"))
    .withColumn("Is_Active", F.lit(None).cast("string"))
    .withColumn("Member_Type", F.lit(None).cast("string"))
    .withColumn("Source_Country", F.lit("AUS"))
)

In [0]:
# --------------------------------------------------
# 2. STANDARDIZE INDIA DATA
# --------------------------------------------------

ind_standard = ind_df.select(
    F.col("Name").alias("Member_Name"),
    F.col("ID").alias("Member_ID"),
    F.col("EnrollmentDate").alias("Enrollment_Date"),
    F.col("Flight Date").alias("Last_Flight_Date"),
    F.col("TierCode").alias("Tier_Code"),
    F.col("DOB").alias("DOB"),
    F.col("Individual or Corporate").alias("Member_Type")
)

ind_standard = (
    ind_standard
    .withColumn("Agent_Name", F.lit(None).cast("string"))
    .withColumn("State", F.lit(None).cast("string"))
    .withColumn("Country", F.lit(None).cast("string"))
    .withColumn("Post_Code", F.lit(None).cast("string"))
    .withColumn("Is_Active", F.lit(None).cast("string"))
    .withColumn("Source_Country", F.lit("IND"))
)

In [0]:
# --------------------------------------------------
# 3. STANDARDIZE USA DATA
# --------------------------------------------------

usa_standard = usa_df.select(
    F.col("Name").alias("Member_Name"),
    F.col("ID").alias("Member_ID"),
    F.col("EnrollmentDate").alias("Enrollment_Date"),
    F.col("FlightDate").alias("Last_Flight_Date"),
    F.col("TierCode").alias("Tier_Code")
)

usa_standard = (
    usa_standard
    .withColumn("DOB", F.lit(None).cast("string"))
    .withColumn("Agent_Name", F.lit(None).cast("string"))
    .withColumn("State", F.lit(None).cast("string"))
    .withColumn("Country", F.lit(None).cast("string"))
    .withColumn("Post_Code", F.lit(None).cast("string"))
    .withColumn("Is_Active", F.lit(None).cast("string"))
    .withColumn("Member_Type", F.lit(None).cast("string"))
    .withColumn("Source_Country", F.lit("USA"))
)

In [0]:
# --------------------------------------------------
# 4. DEFINE COMMON COLUMN ORDER
# --------------------------------------------------

common_columns = [
    "Member_Name",
    "Member_ID",
    "Enrollment_Date",
    "Last_Flight_Date",
    "Tier_Code",
    "Agent_Name",
    "State",
    "Country",
    "Post_Code",
    "DOB",
    "Is_Active",
    "Member_Type",
    "Source_Country"
]

aus_standard = aus_standard.select(*common_columns)
ind_standard = ind_standard.select(*common_columns)
usa_standard = usa_standard.select(*common_columns)

In [0]:
# --------------------------------------------------
# 5. COMBINE ALL THREE DATASETS
# --------------------------------------------------

member_combined = (
    aus_standard
    .unionByName(ind_standard)
    .unionByName(usa_standard)
)

In [0]:
# --------------------------------------------------
# 6. ADD INGESTION METADATA
# --------------------------------------------------

member_combined = (
    member_combined
    .withColumn("Ingested_At", F.current_timestamp())
)

In [0]:
# --------------------------------------------------
# 7. DISPLAY FINAL DATA
# --------------------------------------------------

print("Australia count:", aus_standard.count())
print("India count:", ind_standard.count())
print("USA count:", usa_standard.count())

print("Total combined records:", member_combined.count())

print("Final schema:")
member_combined.printSchema()

display(member_combined)

Australia count: 3
India count: 3
USA count: 3
Total combined records: 9
Final schema:
root
 |-- Member_Name: string (nullable = true)
 |-- Member_ID: string (nullable = true)
 |-- Enrollment_Date: string (nullable = true)
 |-- Last_Flight_Date: string (nullable = true)
 |-- Tier_Code: string (nullable = true)
 |-- Agent_Name: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Post_Code: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Member_Type: string (nullable = true)
 |-- Source_Country: string (nullable = false)
 |-- Ingested_At: timestamp (nullable = false)



Member_Name,Member_ID,Enrollment_Date,Last_Flight_Date,Tier_Code,Agent_Name,State,Country,Post_Code,DOB,Is_Active,Member_Type,Source_Country,Ingested_At
Mike,1,2022-05-11 00:00:00,2022-08-01 00:00:00,PLT,null,null,null,null,NULL,null,null,AUS,2026-11-21T07:13:27.796Z
Jonnathan,2,2021-13-13,2022-01-05 00:00:00,GLD,null,null,null,null,1997-12-13 00:00:00,null,null,AUS,2026-11-21T07:13:27.796Z
Cristina,3,2022-03-12 00:00:00,2022-03-20 00:00:00,SLV,null,null,null,null,1998-03-12 00:00:00,null,null,AUS,2026-11-21T07:13:27.796Z
Vikas,1,2022-01-01 00:00:00,2022-06-15 00:00:00,SLV,null,null,null,null,1998-12-01 00:00:00,null,I,IND,2026-11-21T07:13:27.796Z
Rahul,2,2022-03-05 00:00:00,2022-03-10 00:00:00,GLD,null,null,null,null,1982-08-13 00:00:00,null,C,IND,2026-11-21T07:13:27.796Z
Sameer,3,2022-02-20 00:00:00,2022-02-25 00:00:00,GLD,null,null,null,null,1952-08-13 00:00:00,null,I,IND,2026-11-21T07:13:27.796Z
Sam,1,6152022,8202022,PLT,null,null,null,null,null,null,null,USA,2026-11-21T07:13:27.796Z
John,2,1052022,1152022,SLV,null,null,null,null,null,null,null,USA,2026-11-21T07:13:27.796Z
Mike,3,12282021,12302021,GLD,null,null,null,null,null,null,null,USA,2026-11-21T07:13:27.796Z


CHECKS


In [0]:
# 1. Check total records
print("Total records:", member_combined.count())

Total records: 9


In [0]:
# 2. Check records by source country
display(
    member_combined
    .groupBy("Source_Country")
    .count()
)

Source_Country,count
AUS,3
IND,3
USA,3


In [0]:
# 3. Check duplicate Member IDs
display(
    member_combined
    .groupBy("Member_ID")
    .count()
    .filter(F.col("count") > 1)
)


Member_ID,count
1,3
2,3
3,3


In [0]:
# 4. Check missing values
display(
    member_combined.select([
        F.count(
            F.when(
                F.col(c).isNull() | (F.trim(F.col(c)) == ""),
                c
            )
        ).alias(c)
        for c in member_combined.columns
    ])
)

Member_Name,Member_ID,Enrollment_Date,Last_Flight_Date,Tier_Code,Agent_Name,State,Country,Post_Code,DOB,Is_Active,Member_Type,Source_Country,Ingested_At
0,0,0,0,0,9,9,9,9,3,9,6,0,0


In [0]:
# 5. Preview combined data
display(member_combined)

Member_Name,Member_ID,Enrollment_Date,Last_Flight_Date,Tier_Code,Agent_Name,State,Country,Post_Code,DOB,Is_Active,Member_Type,Source_Country,Ingested_At
Mike,1,2022-05-11 00:00:00,2022-08-01 00:00:00,PLT,null,null,null,null,NULL,null,null,AUS,2026-11-21T07:16:24.510Z
Jonnathan,2,2021-13-13,2022-01-05 00:00:00,GLD,null,null,null,null,1997-12-13 00:00:00,null,null,AUS,2026-11-21T07:16:24.510Z
Cristina,3,2022-03-12 00:00:00,2022-03-20 00:00:00,SLV,null,null,null,null,1998-03-12 00:00:00,null,null,AUS,2026-11-21T07:16:24.510Z
Vikas,1,2022-01-01 00:00:00,2022-06-15 00:00:00,SLV,null,null,null,null,1998-12-01 00:00:00,null,I,IND,2026-11-21T07:16:24.510Z
Rahul,2,2022-03-05 00:00:00,2022-03-10 00:00:00,GLD,null,null,null,null,1982-08-13 00:00:00,null,C,IND,2026-11-21T07:16:24.510Z
Sameer,3,2022-02-20 00:00:00,2022-02-25 00:00:00,GLD,null,null,null,null,1952-08-13 00:00:00,null,I,IND,2026-11-21T07:16:24.510Z
Sam,1,6152022,8202022,PLT,null,null,null,null,null,null,null,USA,2026-11-21T07:16:24.510Z
John,2,1052022,1152022,SLV,null,null,null,null,null,null,null,USA,2026-11-21T07:16:24.510Z
Mike,3,12282021,12302021,GLD,null,null,null,null,null,null,null,USA,2026-11-21T07:16:24.510Z


Write data into the Bronze layer


In [0]:
bronze_path = "abfss://bronze@incubytedatalake01.dfs.core.windows.net/member_data/"

In [0]:
#Write the combined DataFrame
(
    member_combined.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_path)
)

print("Member data successfully written to Bronze!")

Member data successfully written to Bronze!


In [0]:
#read bronze data

bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

display(bronze_df)

Member_Name,Member_ID,Enrollment_Date,Last_Flight_Date,Tier_Code,Agent_Name,State,Country,Post_Code,DOB,Is_Active,Member_Type,Source_Country,Ingested_At
Mike,1,2022-05-11 00:00:00,2022-08-01 00:00:00,PLT,null,null,null,null,NULL,null,null,AUS,2026-11-21T07:18:39.239Z
Jonnathan,2,2021-13-13,2022-01-05 00:00:00,GLD,null,null,null,null,1997-12-13 00:00:00,null,null,AUS,2026-11-21T07:18:39.239Z
Cristina,3,2022-03-12 00:00:00,2022-03-20 00:00:00,SLV,null,null,null,null,1998-03-12 00:00:00,null,null,AUS,2026-11-21T07:18:39.239Z
Vikas,1,2022-01-01 00:00:00,2022-06-15 00:00:00,SLV,null,null,null,null,1998-12-01 00:00:00,null,I,IND,2026-11-21T07:18:39.239Z
Rahul,2,2022-03-05 00:00:00,2022-03-10 00:00:00,GLD,null,null,null,null,1982-08-13 00:00:00,null,C,IND,2026-11-21T07:18:39.239Z
Sameer,3,2022-02-20 00:00:00,2022-02-25 00:00:00,GLD,null,null,null,null,1952-08-13 00:00:00,null,I,IND,2026-11-21T07:18:39.239Z
Sam,1,6152022,8202022,PLT,null,null,null,null,null,null,null,USA,2026-11-21T07:18:39.239Z
John,2,1052022,1152022,SLV,null,null,null,null,null,null,null,USA,2026-11-21T07:18:39.239Z
Mike,3,12282021,12302021,GLD,null,null,null,null,null,null,null,USA,2026-11-21T07:18:39.239Z


In [0]:
#Verify the Bronze records
print("Bronze record count:", bronze_df.count())

print("Bronze schema:")
bronze_df.printSchema()


Bronze record count: 9
Bronze schema:
root
 |-- Member_Name: string (nullable = true)
 |-- Member_ID: string (nullable = true)
 |-- Enrollment_Date: string (nullable = true)
 |-- Last_Flight_Date: string (nullable = true)
 |-- Tier_Code: string (nullable = true)
 |-- Agent_Name: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Post_Code: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Member_Type: string (nullable = true)
 |-- Source_Country: string (nullable = true)
 |-- Ingested_At: timestamp (nullable = true)

